In [1]:
import json
import logging
from pathlib import Path
from typing import List, Dict, Set
from dataclasses import dataclass
from tqdm import tqdm

In [2]:
# Configuration
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

OVERLAP_THRESHOLD = 0.75  # 75% overlap 
DATA_DIR = Path("../data")


@dataclass
class FilterStats:
    """Statistics"""
    total_chunks: int = 0
    biblical_chunks: int = 0
    filtered_chunks: int = 0
    
    @property
    def biblical_percentage(self) -> float:
        return (self.biblical_chunks / self.total_chunks * 100) if self.total_chunks > 0 else 0.0
    
    @property
    def retention_rate(self) -> float:
        return (self.filtered_chunks / self.total_chunks) if self.total_chunks > 0 else 0.0


class BiblicalFilter:
    """
    Filter chunks based on Overlap Coefficient with Vulgate.
    """
    
    def __init__(self, vulgate_chunks: List[Dict], threshold: float = OVERLAP_THRESHOLD):
        self.threshold = threshold
        
        # Pre-compute Vulgate lemma sets for fast lookup
        logger.info(f"Pre-computing Vulgate lemma sets from {len(vulgate_chunks)} verses...")
        self.vulgate_sets = {}
        
        for verse in tqdm(vulgate_chunks, desc="Building Vulgate index"):
            verse_id = verse["chunk_id"]
            lemmas = verse.get("lemmatized", [])
            
            # Remove punctuation and convert to lowercase set
            clean_lemmas = [l.lower() for l in lemmas if l and l.isalpha()]
            self.vulgate_sets[verse_id] = set(clean_lemmas)
        
        logger.info(f"✓ Indexed {len(self.vulgate_sets)} Vulgate verses\n")
    
    def overlap_coefficient(self, set1: Set[str], set2: Set[str]) -> float:
        """
        Compute Overlap Coefficient (Szymkiewicz–Simpson coefficient).
        
        Args:
            set1: First set of tokens
            set2: Second set of tokens
        
        Returns:
            Overlap coefficient (0.0 to 1.0)
        """
        if not set1 or not set2:
            return 0.0
        
        intersection = len(set1 & set2)
        min_size = min(len(set1), len(set2))
        
        return intersection / min_size
    
    def filter_chunk(self, chunk_tokens: List[str]) -> Dict:
        """
        Check if chunk contains pure/near-pure biblical quotation.
        
        Args:
            chunk_tokens: List of tokens (lemmas for BDC, words for PSC)
        
        Returns:
            Dict with:
                - is_biblical: bool
                - max_overlap: float
                - matching_verse: str (if biblical)
        """
        # Clean tokens (remove punctuation, lowercase)
        clean_tokens = [t.lower() for t in chunk_tokens if t and t.isalpha()]
        
        if not clean_tokens:
            return {
                "is_biblical": False,
                "max_overlap": 0.0,
                "matching_verse": None
            }
        
        chunk_set = set(clean_tokens)
        
        # Find max overlap across all Vulgate verses
        max_overlap = 0.0
        matching_verse = None
        
        for verse_id, verse_set in self.vulgate_sets.items():
            overlap = self.overlap_coefficient(chunk_set, verse_set)
            
            if overlap > max_overlap:
                max_overlap = overlap
                matching_verse = verse_id
        
        is_biblical = max_overlap >= self.threshold
        
        return {
            "is_biblical": is_biblical,
            "max_overlap": round(max_overlap, 4),
            "matching_verse": matching_verse if is_biblical else None
        }
    
    def filter_corpus(self, chunks: List[Dict], corpus_name: str, use_lemmas: bool = True) -> Dict:
        """
        Filter entire corpus and return filtered results.
        
        Args:
            chunks: List of chunk dictionaries
            corpus_name: Name for logging (BDC/PSC)
            use_lemmas: If True, use 'lemmatized' field; else tokenize 'text'
        
        Returns:
            Dict with filtered_chunks, biblical_chunks, and stats
        """
        logger.info(f"Filtering {corpus_name} ({len(chunks)} chunks)...")
        logger.info(f"Using {'lemmatized' if use_lemmas else 'text'} field")
        logger.info(f"Overlap threshold: {self.threshold}")
        logger.info(f"Method: Overlap Coefficient (Szymkiewicz–Simpson)")
        logger.info(f"Filters chunks where overlap ≥{self.threshold*100:.0f}% with any Bible verse\n")
        
        filtered_chunks = []
        biblical_chunks = []
        
        for chunk in tqdm(chunks, desc=f"Filtering {corpus_name}"):
            # Get tokens based on corpus type
            if use_lemmas:
                tokens = chunk.get("lemmatized", [])
            else:
                # PSC: tokenize text on whitespace
                tokens = chunk.get("text", "").split()
            
            # Compute biblical overlap
            result = self.filter_chunk(tokens)
            
            # Add filtering metadata to chunk
            chunk_with_metadata = chunk.copy()
            chunk_with_metadata.update(result)
            
            # Categorise
            if result["is_biblical"]:
                biblical_chunks.append(chunk_with_metadata)
            else:
                filtered_chunks.append(chunk_with_metadata)
        
        # Compute statistics
        stats = FilterStats(
            total_chunks=len(chunks),
            biblical_chunks=len(biblical_chunks),
            filtered_chunks=len(filtered_chunks)
        )
        
        # Log results
        logger.info(f"\n{corpus_name} FILTERING RESULTS:")
        logger.info(f"  Total chunks: {stats.total_chunks:,}")
        logger.info(f"  Biblical (filtered out): {stats.biblical_chunks:,} ({stats.biblical_percentage:.1f}%)")
        logger.info(f"  Retained: {stats.filtered_chunks:,} ({stats.retention_rate*100:.1f}%)")
        logger.info("")
        
        return {
            "filtered_chunks": filtered_chunks,
            "biblical_chunks": biblical_chunks,
            "stats": {
                "total": stats.total_chunks,
                "biblical": stats.biblical_chunks,
                "retained": stats.filtered_chunks,
                "biblical_percentage": stats.biblical_percentage,
                "retention_rate": stats.retention_rate,
            }
        }


def load_chunks(filepath: Path) -> List[Dict]:
    """Load chunks from JSON file."""
    logger.info(f"Loading {filepath.name}...")
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    chunks = data.get("chunks", [])
    logger.info(f"✓ Loaded {len(chunks):,} chunks\n")
    return chunks


def save_filtered_data(output_path: Path, filtered_data: Dict, corpus_name: str):
    """Save filtered chunks to JSON."""
    output_data = {
        "metadata": {
            "corpus": corpus_name,
            "filtering": {
                "method": "Overlap Coefficient (Szymkiewicz-Simpson)",
                "threshold": OVERLAP_THRESHOLD,
                "description": f"Filters chunks where overlap ≥{OVERLAP_THRESHOLD*100:.0f}% with any Bible verse",
                "formula": "|chunk ∩ verse| / min(|chunk|, |verse|)",
                "total_chunks": filtered_data["stats"]["total"],
                "biblical_chunks": filtered_data["stats"]["biblical"],
                "retained_chunks": filtered_data["stats"]["retained"],
                "retention_rate": filtered_data["stats"]["retention_rate"],
            }
        },
        "filtered_chunks": filtered_data["filtered_chunks"],
        "biblical_chunks": filtered_data["biblical_chunks"],
    }
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    
    logger.info(f"✓ Saved to {output_path}\n")


def main():
    """Main filtering pipeline."""
    
    logger.info("Overlap Coefficient")
    
    # Define paths
    vulgate_path = DATA_DIR / "vulgate_chunks.json"
    bdc_path = DATA_DIR / "VD_bdc_chunks.json"
    psc_path = DATA_DIR / "VD_psc_chunks.json"
    
    output_dir = DATA_DIR / "filtered"
    output_dir.mkdir(exist_ok=True)
    
    # Load data
    vulgate_chunks = load_chunks(vulgate_path)
    bdc_chunks = load_chunks(bdc_path)
    psc_chunks = load_chunks(psc_path)
    
    # Initialize filter
    biblical_filter = BiblicalFilter(vulgate_chunks, threshold=OVERLAP_THRESHOLD)
    
    # Filter BDC (use lemmatized field)
    logger.info("Filtering BDC")
    
    bdc_filtered = biblical_filter.filter_corpus(
        bdc_chunks, 
        corpus_name="BDC",
        use_lemmas=True 
    )
    
    save_filtered_data(
        output_dir / "bdc_filtered.json",
        bdc_filtered,
        "BDC"
    )
    
    # Filter PSC (tokenize text on whitespace)
    logger.info("Filtering PSC (Patristic Source Corpus)")
    
    psc_filtered = biblical_filter.filter_corpus(
        psc_chunks,
        corpus_name="PSC", 
        use_lemmas=False  
    )
    
    save_filtered_data(
        output_dir / "psc_filtered.json",
        psc_filtered,
        "PSC"
    )
    
    # Summary
    logger.info("Filtering done")
    logger.info(f"\nBDC: {bdc_filtered['stats']['retained']:,} / {bdc_filtered['stats']['total']:,} retained")
    logger.info(f"PSC: {psc_filtered['stats']['retained']:,} / {psc_filtered['stats']['total']:,} retained")

if __name__ == "__main__":
    main()

INFO: Overlap Coefficient
INFO: Loading vulgate_chunks.json...
INFO: ✓ Loaded 35,809 chunks

INFO: Loading VD_bdc_chunks.json...
INFO: ✓ Loaded 19,466 chunks

INFO: Loading VD_psc_chunks.json...
INFO: ✓ Loaded 46,408 chunks

INFO: Pre-computing Vulgate lemma sets from 35809 verses...
Building Vulgate index: 100%|██████████| 35809/35809 [00:00<00:00, 162850.93it/s]
INFO: ✓ Indexed 35809 Vulgate verses

INFO: Filtering BDC
INFO: Filtering BDC (19466 chunks)...
INFO: Using lemmatized field
INFO: Overlap threshold: 0.75
INFO: Method: Overlap Coefficient (Szymkiewicz–Simpson)
INFO: Filters chunks where overlap ≥75% with any Bible verse

Filtering BDC: 100%|██████████| 19466/19466 [05:52<00:00, 55.21it/s]
INFO: 
BDC FILTERING RESULTS:
INFO:   Total chunks: 19,466
INFO:   Biblical (filtered out): 6,736 (34.6%)
INFO:   Retained: 12,730 (65.4%)
INFO: 
INFO: ✓ Saved to ../data/filtered/bdc_filtered.json

INFO: Filtering PSC (Patristic Source Corpus)
INFO: Filtering PSC (46408 chunks)...
INFO: Us